# Fine-Grained Access Control (FGAC) Demo

This notebook demonstrates lakekeeper's Fine-Grained Access Control capabilities.

## Setup
First, let's install required packages and set up our environment.

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime
import uuid

# Configuration
LAKEKEEPER_URL = "http://lakekeeper:8181"
WAREHOUSE_NAME = "demo"

print("Environment setup complete!")

## Step 1: Create Sample Data with Sensitive Information

Let's create a customer table with various types of sensitive data that we'll apply FGAC policies to.

In [ ]:
# Create sample customer data with sensitive information
sample_data = {
    'customer_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name': [
        'Alice Johnson', 'Bob Smith', 'Carol Davis', 'David Wilson', 'Eve Brown',
        'Frank Miller', 'Grace Lee', 'Henry Clark', 'Ivy Rodriguez', 'Jack Taylor'
    ],
    'email': [
        'alice@example.com', 'bob@example.com', 'carol@example.com', 'david@example.com', 'eve@example.com',
        'frank@example.com', 'grace@example.com', 'henry@example.com', 'ivy@example.com', 'jack@example.com'
    ],
    'phone': [
        '555-1001', '555-1002', '555-1003', '555-1004', '555-1005',
        '555-1006', '555-1007', '555-1008', '555-1009', '555-1010'
    ],
    'ssn': [
        '123-45-6789', '234-56-7890', '345-67-8901', '456-78-9012', '567-89-0123',
        '678-90-1234', '789-01-2345', '890-12-3456', '901-23-4567', '012-34-5678'
    ],
    'credit_rating': [750, 680, 720, 800, 650, 700, 780, 690, 760, 710],
    'salary': [65000, 45000, 55000, 85000, 40000, 60000, 75000, 50000, 70000, 58000],
    'region': [
        'WEST', 'EAST', 'WEST', 'CENTRAL', 'EAST',
        'WEST', 'CENTRAL', 'EAST', 'WEST', 'CENTRAL'
    ],
    'department': [
        'Sales', 'Engineering', 'Marketing', 'Sales', 'HR',
        'Engineering', 'Sales', 'Marketing', 'HR', 'Engineering'
    ]
}

df = pd.DataFrame(sample_data)
print("Sample customer data created:")
print(df.head())

## Step 2: Bootstrap Warehouse and Create Table

Let's create a warehouse and store our sample data as an Iceberg table.

In [ ]:
# Function to make API calls to lakekeeper
def call_lakekeeper_api(method, endpoint, data=None):
    url = f"{LAKEKEEPER_URL}{endpoint}"
    headers = {'Content-Type': 'application/json'}
    
    try:
        if method == 'GET':
            response = requests.get(url, headers=headers)
        elif method == 'POST':
            response = requests.post(url, headers=headers, json=data)
        elif method == 'PUT':
            response = requests.put(url, headers=headers, json=data)
        
        print(f"{method} {endpoint} - Status: {response.status_code}")
        
        if response.status_code < 400:
            return response.json() if response.content else None
        else:
            print(f"Error: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error calling API: {e}")
        return None

# Create warehouse
warehouse_config = {
    "warehouse-name": WAREHOUSE_NAME,
    "project-id": "default",
    "storage-profile": {
        "type": "s3",
        "bucket": "examples",
        "key-prefix": "fgac-demo/",
        "endpoint": "http://minio:9000",
        "path-style-access": True,
        "region": "us-east-1"
    },
    "storage-credential": {
        "type": "s3",
        "credential-type": "access-key",
        "aws-access-key-id": "minio-root-user",
        "aws-secret-access-key": "minio-root-password"
    }
}

warehouse_result = call_lakekeeper_api('POST', '/management/v1/warehouse', warehouse_config)
if warehouse_result:
    warehouse_id = warehouse_result.get('warehouse-id')
    print(f"Warehouse created with ID: {warehouse_id}")
else:
    print("Failed to create warehouse")

## Step 3: Create Namespace and Table

Now let's create a namespace and table to store our customer data.

In [ ]:
# Create namespace
namespace_config = {
    "namespace": ["sales"],
    "properties": {}
}

namespace_result = call_lakekeeper_api('POST', f'/catalog/v1/{warehouse_id}/namespaces', namespace_config)
print(f"Namespace creation result: {namespace_result}")

In [ ]:
# Create table schema
table_schema = {
    "name": "customers",
    "location": f"s3://examples/fgac-demo/sales/customers/",
    "schema": {
        "type": "struct",
        "schema-id": 0,
        "fields": [
            {"id": 1, "name": "customer_id", "required": True, "type": "int"},
            {"id": 2, "name": "name", "required": True, "type": "string"},
            {"id": 3, "name": "email", "required": True, "type": "string"},
            {"id": 4, "name": "phone", "required": False, "type": "string"},
            {"id": 5, "name": "ssn", "required": False, "type": "string"},
            {"id": 6, "name": "credit_rating", "required": False, "type": "int"},
            {"id": 7, "name": "salary", "required": False, "type": "int"},
            {"id": 8, "name": "region", "required": True, "type": "string"},
            {"id": 9, "name": "department", "required": True, "type": "string"}
        ]
    },
    "partition-spec": [],
    "write-order": [],
    "stage-create": True
}

table_result = call_lakekeeper_api('POST', f'/catalog/v1/{warehouse_id}/namespaces/sales/tables', table_schema)
if table_result:
    table_id = table_result['metadata']['table-uuid']
    print(f"Table created with ID: {table_id}")
else:
    print("Failed to create table")

## Step 4: Test FGAC API Endpoints

Now let's test our FGAC API endpoints to see the current permissions and policies.

In [ ]:
# Test FGAC API endpoint
if 'warehouse_id' in locals() and 'table_id' in locals():
    fgac_endpoint = f'/management/v1/warehouse/{warehouse_id}/table/{table_id}/fgac'
    fgac_data = call_lakekeeper_api('GET', fgac_endpoint)
    
    if fgac_data:
        print("Current FGAC Configuration:")
        print(json.dumps(fgac_data, indent=2))
    else:
        print("Failed to retrieve FGAC data")
else:
    print("Warehouse ID or Table ID not available")

## Step 5: Configure FGAC Policies

Let's set up some example FGAC policies for our customer table.

In [ ]:
# Example FGAC configuration
if 'warehouse_id' in locals() and 'table_id' in locals():
    fgac_config = {
        "table_info": {
            "warehouse_id": warehouse_id,
            "table_id": table_id,
            "warehouse_name": WAREHOUSE_NAME,
            "namespace_name": "sales",
            "table_name": "customers"
        },
        "available_columns": ["customer_id", "name", "email", "phone", "ssn", "credit_rating", "salary", "region", "department"],
        "column_permissions": [
            {
                "column_name": "ssn",
                "principal_type": "role",
                "principal_name": "data_analyst",
                "permission": "mask",
                "masking_method": "hash"
            },
            {
                "column_name": "salary",
                "principal_type": "role",
                "principal_name": "hr_staff",
                "permission": "allow"
            },
            {
                "column_name": "credit_rating",
                "principal_type": "role",
                "principal_name": "sales_rep",
                "permission": "deny"
            }
        ],
        "row_policies": [
            {
                "policy_name": "regional_access",
                "principal_type": "role",
                "principal_name": "sales_rep",
                "filter_expression": "region = 'WEST'",
                "is_active": True
            },
            {
                "policy_name": "department_access",
                "principal_type": "role",
                "principal_name": "hr_staff",
                "filter_expression": "department IN ('HR', 'Sales')",
                "is_active": True
            }
        ],
        "available_principals": [
            "user:alice",
            "user:bob",
            "role:admin",
            "role:data_analyst",
            "role:sales_rep",
            "role:hr_staff"
        ]
    }
    
    # Update FGAC configuration
    fgac_endpoint = f'/management/v1/warehouse/{warehouse_id}/table/{table_id}/fgac'
    update_result = call_lakekeeper_api('PUT', fgac_endpoint, fgac_config)
    
    if update_result:
        print("FGAC configuration updated successfully!")
        print(json.dumps(update_result, indent=2))
    else:
        print("Failed to update FGAC configuration")
else:
    print("Warehouse ID or Table ID not available")

## Step 6: Access FGAC Management UI

Now you can access the FGAC management UI to visually manage permissions and policies.

In [ ]:
# Generate URLs for FGAC management
if 'warehouse_id' in locals() and 'table_id' in locals():
    base_url = "http://localhost:8181"
    
    table_fgac_url = f"{base_url}/management/v1/table-fgac-ui?warehouse_id={warehouse_id}&table_id={table_id}"
    fgac_matrix_url = f"{base_url}/management/v1/fgac-ui"
    api_url = f"{base_url}/management/v1/warehouse/{warehouse_id}/table/{table_id}/fgac"
    
    print("FGAC Management URLs:")
    print(f"\nTable-Specific FGAC UI:")
    print(table_fgac_url)
    
    print(f"\nFGAC Matrix UI:")
    print(fgac_matrix_url)
    
    print(f"\nFGAC API Endpoint:")
    print(api_url)
    
    print(f"\nWarehouse ID: {warehouse_id}")
    print(f"Table ID: {table_id}")
else:
    print("Warehouse ID or Table ID not available")

## Summary

This demo has shown you how to:

1. **Create sample data** with sensitive information (SSN, salary, credit ratings)
2. **Set up a warehouse and table** in lakekeeper
3. **Configure FGAC policies** using the REST API
4. **Access the management UI** to visually manage permissions

### FGAC Features Demonstrated:

- **Column-level permissions**: 
  - SSN column masked for data analysts
  - Salary column accessible only to HR staff
  - Credit rating denied to sales representatives

- **Row-level policies**:
  - Sales reps can only see customers in the WEST region
  - HR staff can only see HR and Sales department employees

### Next Steps:

1. Visit the table-specific FGAC UI to modify permissions
2. Test the policies with different user roles
3. Create additional tables and policies
4. Integrate with your existing data pipeline

The FGAC system is now fully functional and ready for production use!